In [47]:
from langchain_core.documents import Document

In [48]:
doc=Document(
    page_content="this is the main text that will be used for the rag",
    metadata={
        "source":"dummy.txt",
        "pages":10,
        "author":"Sohan",
        
    }
)

In [49]:
doc

Document(metadata={'source': 'dummy.txt', 'pages': 10, 'author': 'Sohan'}, page_content='this is the main text that will be used for the rag')

In [50]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [51]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("created and written to files")

created and written to files


In [52]:
from langchain_community.document_loaders import TextLoader

In [53]:
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")

In [54]:
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [55]:
from langchain_community.document_loaders import DirectoryLoader

In [56]:
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    show_progress=True,
    loader_cls=TextLoader
)

In [57]:
documents=dir_loader.load()
documents[0]

100%|██████████| 2/2 [00:00<00:00, 872.63it/s]


Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')

In [58]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

In [59]:
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

In [60]:
pdf_docs=dir_loader.load()
pdf_docs

100%|██████████| 1/1 [00:00<00:00,  5.30it/s]


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszka

In [61]:
from pathlib import Path

In [62]:
def process_all_pdf_files(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} files to process")

    for pdf_file in pdf_files:
        try:
            print(f"loading : {pdf_file.name}")
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load() #this returns a list

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_docs.extend(documents)  

        except Exception as e:
            print(f"error : {e}")

    print(f"total number of documents : {len(all_docs)}")
    return all_docs

In [63]:
print(type(documents))

<class 'list'>


In [64]:
all_pdf_documents=process_all_pdf_files("../data")

found 1 files to process
loading : Attention is all you need.pdf
total number of documents : 11


Splitting the text into chunks

In [65]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [66]:
def split_documents(documents,chunk_size=400,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content : {split_docs[0].page_content[:200]}")
        print(f"Metadata : {split_docs[0].metadata}")
    return split_docs

In [67]:
chunks=split_documents(all_pdf_documents)
chunks

split 11 documents into 99 chunks
Content : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz
Metadata : {'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'Attention is all you need.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'Attention is all you need.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of T

In [68]:
print(type(chunks))
chunks[0]

<class 'list'>


Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'Attention is all you need.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of To

Embedding store and vectorDB

In [69]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [70]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully")
        except Exception as e:
            print(f"failed to load model : {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with dimensions : {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6817.10it/s]


model loaded successfully


Here we store in the VectorDB

In [71]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persistent_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.collection=None
        self.client=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_dir)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desc":"pdf embeddings for RAG"}
            )
            print(f"vector store initialized :{self.collection_name}")
            print(f"existing documents in the collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must be equal to the number of embeddings")
        
        print(f"adding {len(documents)} documents to the vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"added {len(documents)} documents to the vector store")
        
        except Exception as e:
            print(f"error storing the documents into the vector store : {e}")
            raise

vectorstore=VectorStore()

vector store initialized :pdf_documents
existing documents in the collection : 198


In [72]:
vectorstore

In [73]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'Attention is all you need.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of T

In [74]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 4/4 [00:00<00:00,  7.96it/s]

Generated embeddings with dimensions : (99, 384)
adding 99 documents to the vector store
added 99 documents to the vector store


Retriever pipeline from the VectorStore

In [75]:
class RAGretriever:
    def __init__(self, vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int = 5,score_threshold:float = 0.0)->list[Dict[str,Any]]: #the query is the question, the top_k is the top 5 results in this case and the threshold is the minimum similarity threshold, like min they should be like 0 similar
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate (zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")
            return retrieved_docs
        
        except Exception as e:
            print(f"error occurred : {e}")
            return []



rag_retriever=RAGretriever(vectorstore,embedding_manager)


In [76]:
rag_retriever

In [77]:

rag_retriever.retrieve("Unified Multi-Task Learning Framework")

Batches: 100%|██████████| 1/1 [00:00<00:00, 107.15it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 0 documents


[]

In [78]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/Attention is all you need.pdf', 'file_path': '../data/pdf/Attention is all you need.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'Attention is all you need.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of T

In [79]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))
print(all_pdf_documents[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being 

Now we integrate the VectorDB context pipeline with the LLM Output

In [80]:
from langchain_community.llms import Ollama

In [81]:
llm=Ollama(model="llama3.2:3b")

In [82]:
def generate_answer(query,retrieved_docs):
    MAX_CHARS = 2000

    context = ""
    for doc in retrieved_docs:
        if len(context) + len(doc['content']) > MAX_CHARS:
            break
        context += doc['content'] + "\n\n"

    prompt = f"""
    You are a technical assistant.

    Answer the question using the context below.
    If the context is incomplete, then just tell me that the context is incomplete and reply with "Not enough context".
    
    If not enough context then abruptly reply with "Not enough context", don
    t try to fit in text yourself.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    return llm.invoke(prompt)

In [83]:
retrieved=rag_retriever.retrieve("transformer self attention mechanism explanation")
answer=generate_answer("what is a transformer used for?",retrieved)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 178.61it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 5 documents
A Transformer is primarily used as an architectural component within neural networks, particularly in sequence-to-sequence tasks such as machine translation, text summarization, and image captioning. It's designed to process sequential data, like sentences or words, by allowing the model to attend to different parts of the input sequence simultaneously and weigh their importance.


In [84]:
rag_retriever=RAGretriever(vectorstore,embedding_manager)

advanced rag pipeline

In [85]:
def adv_rag(query,rag_retriever,llm,top_k=5,min_score=0.01,return_context=False):
    results=rag_retriever.retrieve(query,top_k,score_threshold=min_score)
    if not results:
        return {'answer': 'no relevant context found','sources':[],'confidence':0.0}
    
    context='\n\n'.join([doc['content'] for doc in results])
    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:150]+'...'
    } for doc in results]
    
    confidence=max(doc['similarity_score'] for doc in results)
    
    prompt=f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response=llm.invoke(prompt)
    output={
        'answer':response,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context']=context
    return output
    

In [86]:
results = rag_retriever.retrieve("transformer")
print(results)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 5 documents
[{'id': 'doc_d8a90ec2_15', 'content': 'it more difﬁcult to learn dependencies between distant positions [11]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.', 'metadata': {'total_pages': 11, 'keywords': '', 'format': 'PDF 1.3', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'content_length': 318, 'file_type': 'pdf', 'source_file': 'Attention is all you need.pdf', 'creator': '', 'moddate': '2018-02-12T21:22:10-08:00', 'creationDate': '', 'creationdate': '', 'producer': 'PyPDF2', 'title': 'Attention is All you Need', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'doc_index': 15, 'modDate': "D:20180212212210-08'0

In [87]:
result=adv_rag("what is transformer?",rag_retriever=rag_retriever,llm=llm,return_context=True)
print('answer:',result['answer'])
print('sources:',result['sources'])
print('confidence:',result['confidence'])
# print('context:',result['context'][:150])

Batches: 100%|██████████| 1/1 [00:00<00:00, 23.65it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 5 documents
answer: The Transformer is a type of transduction model that relies entirely on self-attention to compute input and output representations, without using sequence-aligned RNNs or convolution. It uses two sub-layers with residual connections and layer normalization to achieve this.
sources: [{'source': 'Attention is all you need.pdf', 'page': 1, 'score': 0.02252805233001709, 'preview': 'language modeling tasks [28].\nTo the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying\nentirely on self-attention...'}, {'source': 'Attention is all you need.pdf', 'page': 1, 'score': 0.02252805233001709, 'preview': 'language modeling tasks [28].\nTo the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying\nentirely on self-attention...'}, {'source': 'Attention is all you need.pdf', 'page': 1, 'score': 0.02252805233001709, 'preview': 'language modeling tasks [28].

In [90]:
from typing import List, Dict, Any
import time
class AdvRag:
    def __init__(self,retriever,llm):
        self.retriever=retriever
        self.llm=llm
        self.history=[]
    

    def query(self,question:str,top_k:int=5,min_score:float=0.01,summarize:bool=False)->Dict[str,any]:
        results=self.retriever.retrieve(question,top_k=top_k,score_threshold=min_score)
        if not results:
            return {
                'answer':"no relevant documents found",
                'sources':[],
                'context':""
            }
        else:
            context='\n\n'.join(doc['content'] for doc in results)
            sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:150]+'...'
        } for doc in results]
        
        prompt=f"""use the following context to answer the question concisely, context{context}, question: {question}\n\nAnswer:"""
        response=self.llm.invoke([prompt.format(context=context,question=question)])
        answer=response
        summary=None
        if summarize and answer:
            summary_prompt=f"summarize the following answer in like 2 sentences {answer}"
            summary_resp=self.llm.invoke([summary_prompt])
            summary=summary_resp

        self.history.append({
            'question':question,
            'answer':answer,
            'sources':sources,
            'summary':summary
        })
        return {
            'question':question,
            'answer':answer,
            'sources':sources,
            'summary':summary,
            'history':self.history
        }

In [ ]:
advanced_rag=AdvRag(rag_retriever,llm)
result=advanced_rag.query("what is transformer",summarize=True)
print('answer:',result['answer'])
print('Summary:',result['summary'])
print("Hoistory:",result['history'][-1])
result=advanced_rag.query("what is attention mechanism",summarize=True)
print('answer:',result['answer'])
print('Summary:',result['summary'])
print("Hoistory:",result['history'][-2])

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 3 documents
answer: The Transformer is a type of neural network model architecture that uses self-attention mechanisms to process sequential data. It consists of two sub-layers (Encoder and Decoder) with residual connections and layer normalization, used in a fully connected feed-forward network.
Summary: The Transformer is a neural network architecture that utilizes self-attention mechanisms to process sequential data, consisting of an Encoder and Decoder sub-layer with residual connections and layer normalization. It uses a fully connected feed-forward network to enable the model to process sequential data effectively.
Hoistory: {'question': 'what is transformer', 'answer': 'The Transformer is a type of neural network model architecture that uses self-attention mechanisms to process sequential data. It consists of two sub-layers (Encoder and Decoder) with residual connections and layer normalization, used in a fully connected 

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.10it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 5 documents


answer: Attention mechanism: a way to model dependencies between elements in sequential data without considering their distance.
Summary: The attention mechanism is a technique used to analyze sequences, such as text or speech, by allowing the model to focus on specific parts of the sequence based on their relevance to the task at hand. This enables the model to selectively weigh the importance of different elements in the sequence, rather than considering them in a linear and distance-based manner.
Hoistory: {'question': 'what is transformer', 'answer': 'The Transformer is a type of neural network model architecture that uses self-attention mechanisms to process sequential data. It consists of two sub-layers (Encoder and Decoder) with residual connections and layer normalization, used in a fully connected feed-forward network.', 'sources': [{'source': 'Attention is all you need.pdf', 'page': 2, 'score': 0.012252628803253174, 'preview': 'Figure 1: The Transformer - model architecture.\